# History-rung benchmark: linear, random forest, XGBoost, LightGBM and ANN

Every family is fitted on the same rows, folds and rungs of `12_History_Features`, with the target `y = PL - a` (deviation from the
link's previous-hour mean). Protocol:

* Rungs for the family table: H1 (link history), H4 (+ channel history, clock, SF), H6 (+ earlier SNR, sensors), H7 (+ link level and geometry).
  The full seven-rung ladder is fitted with LightGBM only (it is the cheapest family) and with the linear model.
* Hyperparameters: one Bayesian search per family on rung H4, 20 candidates, scored on the five expanding-window folds of a one-in-ten
  row sample (every tenth row, folds kept), then the chosen configuration is refitted on the full rows at every rung. The static
  notebooks searched 60 candidates on all rows; here the searches are lighter because every family is fitted four times more.
* Predictions: out-of-fold for folds 0-4 (fold k trained on everything before it) and hold-out from the model fitted on all training rows.
* ANN: the same multilayer perceptron as `07_ANN` (Adam, early stopping on the last 10 % of the training rows in time order), averaged over
  three seeds instead of thirty.
* Fade margin: the rule of `09_FADE MARGIN ANALYSIS` (empirical quantile of the out-of-fold residuals; from 98 % upward the larger of that
  and the 3-component GMM tail), checked on the hold-out.
* Unseen link at H4: fit on five links with folds below 4, calibrate the margin on their fold 4, test on the sixth link's hold-out.

Residual files are written as `Residuals/residuals_<family>_<rung>_oof.csv` and `_test.csv` in the schema of the static notebooks.
kNN runs in `14_History_kNN` (GPU kernel), the GRU in `15_History_GRU`; `16_Information_by_Family` collects everything.

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"       # XGBoost on the GPU; the ANN is small and stays on the CPU (see below)
import json
import time
import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.mixture import GaussianMixture
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from skopt import BayesSearchCV
from skopt.space import Real, Integer
import xgboost as xgb
import lightgbm as lgb

RANDOM_STATE = 42
N_JOBS = 32
TARGETS = (0.90, 0.95, 0.99)


In [2]:
train = pd.read_csv("Data_Files/history_train.csv")
test = pd.read_csv("Data_Files/history_test.csv")
rungs = json.load(open("Data_Files/history_rungs.json"))
d = pd.concat([train, test], ignore_index=True)
d["t"] = pd.to_datetime(d["t"], utc=True)
d = d.sort_values("t").reset_index(drop=True)                 # time order: the ANN takes its early-stopping set from the end, residual files feed the block bootstrap
fold = d["fold"].to_numpy()
H = {k.split()[0]: v for k, v in rungs.items()}            # "H1" ... "H7"
BENCH = ["H1", "H4", "H6", "H7"]

# the same rows for every family and rung: an anchor and a complete H7 input
ok = (d["y"].notna() & d[H["H7"]].notna().all(axis=1)).to_numpy()
is_test = fold == 9
cal = ok & (fold >= 0) & (fold < 9)                         # validation folds 0-4: calibration rows
ev = ok & is_test                                           # hold-out rows
PL, a, y = d["PL"].to_numpy(), d["a"].to_numpy(), d["y"].to_numpy()
X_all = d[H["H7"]].copy()
X_all = X_all.fillna(X_all[ok & (fold < 9)].median())      # no missing values reach the models
print(f"rows: training {int((ok & (fold < 9)).sum()):,} | calibration (folds 0-4) {int(cal.sum()):,} | hold-out {int(ev.sum()):,} of {int(is_test.sum()):,}")


rows: training 2,126,679 | calibration (folds 0-4) 1,773,285 | hold-out 531,983 of 532,055


In [3]:
def rmse(e):
    return float(np.sqrt(np.mean(np.square(e))))

def mixture_quantile(w, mu, sd, q):
    lo, hi = float((mu - 10 * sd).min()), float((mu + 10 * sd).max())
    for _ in range(200):
        mid = 0.5 * (lo + hi)
        lo, hi = (mid, hi) if float((w * norm.cdf((mid - mu) / sd)).sum()) < q else (lo, mid)
    return 0.5 * (lo + hi)

def margins(e_cal, e_test):
    g = GaussianMixture(n_components=3, covariance_type="full", random_state=RANDOM_STATE).fit(e_cal.reshape(-1, 1))
    w, mu, sd = g.weights_, g.means_.ravel(), np.sqrt(g.covariances_.reshape(-1))
    out = {}
    for rho in TARGETS:
        emp = float(np.quantile(e_cal, rho))
        fm = max(emp, mixture_quantile(w, mu, sd, rho)) if rho >= 0.98 else emp
        out[f"margin {rho:.2f}"], out[f"reached {rho:.2f}"] = fm, float((e_test <= fm).mean())
    return out

def oof_and_test(make, cols):
    """Out-of-fold predictions of y for folds 0-4 and hold-out predictions from the model fitted on all training rows."""
    X = X_all[cols].to_numpy()
    p = np.full(len(d), np.nan)
    for k in range(5):
        tr, va = ok & (fold < k), ok & (fold == k)
        p[va] = make().fit(X[tr], y[tr]).predict(X[va])
    tr = ok & (fold < 9)
    model = make().fit(X[tr], y[tr])
    p[ev] = model.predict(X[ev])
    return p, model

def save_residuals(name, p):
    for split, mask in (("oof", cal), ("test", ev)):
        pd.DataFrame({"model": name, "split": split, "row_id": np.flatnonzero(mask), "fold": fold[mask], "time": d["t"].to_numpy()[mask],
                      "device_id": d["device_id"].to_numpy()[mask], "PL_true": PL[mask], "PL_pred": (a + p)[mask], "resid_db": (y - p)[mask]}
                     ).to_csv(f"Residuals/residuals_{name}_{split}.csv", index=False)

def evaluate(name, p):
    e = y - p
    row = {"hold-out RMSE": rmse(e[ev]), "hold-out MAE": float(np.abs(e[ev]).mean()), "OOF RMSE": rmse(e[cal])}
    row.update(margins(e[cal], e[ev]))
    return row

os.makedirs("Residuals", exist_ok=True)
os.makedirs("CV_Results", exist_ok=True)
results = {}


In [4]:
# References that need no model: the per-link training mean (the static floor) and the anchor itself
for name, p in (("link mean", d["devmean"].to_numpy() - a), ("previous-hour mean", np.zeros(len(d))), ("previous packet", d["r_lag1"].to_numpy())):
    results[(name, "-")] = evaluate(name, p)
pd.DataFrame(results).T.round(3)


,,hold-out RMSE,hold-out MAE,OOF RMSE,margin 0.90,reached 0.90,margin 0.95,reached 0.95,margin 0.99,reached 0.99
link mean,-,5.035,4.015,6.123,8.017,0.960,10.331,0.985,19.037,0.998
previous-hour mean,-,2.712,2.046,3.464,3.815,0.917,5.309,0.967,12.009,0.999
previous packet,-,3.044,2.241,4.363,4.000,0.939,6.000,0.984,12.485,0.999


In [5]:
# One light Bayesian search per family on rung H4, every tenth row, folds kept
sub = np.flatnonzero(ok & (fold < 9))[::10]
X_sub, y_sub, fold_sub = X_all[H["H4"]].to_numpy()[sub], y[sub], fold[sub]
cv_sub = [(np.flatnonzero(fold_sub < k), np.flatnonzero(fold_sub == k)) for k in range(5)]

def search(name, estimator, space, n_jobs):
    t0 = time.time()
    bs = BayesSearchCV(estimator, space, n_iter=20, cv=cv_sub, scoring="neg_root_mean_squared_error", n_jobs=n_jobs, n_points=2,
                       random_state=RANDOM_STATE, refit=False)
    bs.fit(X_sub, y_sub)
    pd.DataFrame(bs.cv_results_).to_csv(f"CV_Results/history_{name}_search.csv", index=False)
    print(f"{name}: best CV RMSE {-bs.best_score_:.3f} dB on the sample after {time.time() - t0:.0f} s -> {bs.best_params_}")
    return dict(bs.best_params_)

best = {}
best["RF"] = search("RF", RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=4),
                    {"max_depth": Integer(4, 16), "min_samples_leaf": Integer(20, 500), "max_features": Real(0.3, 0.9), "max_samples": Real(0.5, 0.9)}, n_jobs=8)
best["XGB"] = search("XGB", xgb.XGBRegressor(device="cuda", tree_method="hist", random_state=RANDOM_STATE, n_jobs=1),
                     {"max_depth": Integer(3, 10), "learning_rate": Real(0.01, 0.2, prior="log-uniform"), "n_estimators": Integer(200, 1500),
                      "min_child_weight": Real(1, 200, prior="log-uniform"), "subsample": Real(0.6, 1.0), "colsample_bytree": Real(0.5, 1.0),
                      "reg_lambda": Real(0.1, 30, prior="log-uniform")}, n_jobs=1)
best["LGBM"] = search("LGBM", lgb.LGBMRegressor(subsample_freq=1, random_state=RANDOM_STATE, n_jobs=4, verbose=-1),
                      {"num_leaves": Integer(15, 255), "learning_rate": Real(0.01, 0.2, prior="log-uniform"), "n_estimators": Integer(200, 1500),
                       "min_child_samples": Integer(20, 1000), "subsample": Real(0.6, 1.0), "colsample_bytree": Real(0.5, 1.0),
                       "reg_lambda": Real(0.1, 30, prior="log-uniform")}, n_jobs=8)
json.dump(best, open("CV_Results/history_best_params.json", "w"), indent=1, default=float)


RF: best CV RMSE 2.743 dB on the sample after 852 s -> OrderedDict({'max_depth': 16, 'max_features': 0.7068734006537638, 'max_samples': 0.9, 'min_samples_leaf': 20})


/home/nahshon/envs/ml001/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [16:13:05] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)


XGB: best CV RMSE 2.708 dB on the sample after 299 s -> OrderedDict({'colsample_bytree': 0.9064210520153546, 'learning_rate': 0.015807655334598596, 'max_depth': 9, 'min_child_weight': 200.0, 'n_estimators': 1427, 'reg_lambda': 0.5809825635615318, 'subsample': 0.6203536863020367})


LGBM: best CV RMSE 2.716 dB on the sample after 95 s -> OrderedDict({'colsample_bytree': 0.9061979941786817, 'learning_rate': 0.0167343965754388, 'min_child_samples': 606, 'n_estimators': 1244, 'num_leaves': 141, 'reg_lambda': 0.17236683473219908, 'subsample': 0.9023202131343526})


In [6]:
# Linear, RF, XGBoost and LightGBM on the benchmark rungs; the full ladder for the linear model and LightGBM
class Linear:
    def __init__(self):
        self.scaler, self.model = StandardScaler(), Ridge(alpha=1.0)
    def fit(self, X, y):
        self.model.fit(self.scaler.fit_transform(X), y)
        return self
    def predict(self, X):
        return self.model.predict(self.scaler.transform(X))

factories = {
    "Linear": lambda: Linear(),
    "RF": lambda: RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=N_JOBS, **best["RF"]),
    "XGB": lambda: xgb.XGBRegressor(device="cuda", tree_method="hist", random_state=RANDOM_STATE, **best["XGB"]),
    "LGBM": lambda: lgb.LGBMRegressor(subsample_freq=1, random_state=RANDOM_STATE, n_jobs=8, verbose=-1, **best["LGBM"]),   # 8 threads: LightGBM stalls when oversubscribed
}
models = {}
for name, make in factories.items():
    for rung in (list(H) if name in ("Linear", "LGBM") else BENCH):
        t0 = time.time()
        p, models[(name, rung)] = oof_and_test(make, H[rung])
        results[(name, rung)] = evaluate(name, p)
        save_residuals(f"{name}_{rung}", p)
        print(f"{name} {rung}: hold-out RMSE {results[(name, rung)]['hold-out RMSE']:.3f} dB, margin 0.99 {results[(name, rung)]['margin 0.99']:.2f} dB  ({time.time() - t0:.0f} s)")


Linear H1: hold-out RMSE 2.508 dB, margin 0.99 11.79 dB  (19 s)


Linear H2: hold-out RMSE 2.399 dB, margin 0.99 11.64 dB  (19 s)


Linear H3: hold-out RMSE 2.399 dB, margin 0.99 11.64 dB  (19 s)


Linear H4: hold-out RMSE 2.393 dB, margin 0.99 11.71 dB  (19 s)


Linear H5: hold-out RMSE 2.372 dB, margin 0.99 11.37 dB  (20 s)


...

LGBM H3: hold-out RMSE 2.082 dB, margin 0.99 10.52 dB  (169 s)


LGBM H4: hold-out RMSE 1.950 dB, margin 0.99 9.98 dB  (175 s)


LGBM H5: hold-out RMSE 1.821 dB, margin 0.99 10.17 dB  (190 s)


LGBM H6: hold-out RMSE 1.824 dB, margin 0.99 10.20 dB  (205 s)


LGBM H7: hold-out RMSE 1.825 dB, margin 0.99 10.11 dB  (218 s)


In [7]:
# ANN: the multilayer perceptron of 07_ANN, three seeds averaged, early stopping on the last 10 % of the training rows in time order
import tensorflow as tf
from tensorflow import keras
tf.config.set_visible_devices([], "GPU")                    # small network: the CPU is faster than the GPU transfer, and the GPU is XGBoost's

ANN_SEEDS = (0, 1, 2)

def ann_candidates():
    return {(32, 16): None, (64, 32): None, (64, 32, 16): None, (128, 64, 32): None}

class ANN:
    def __init__(self, layers, seeds=ANN_SEEDS):
        self.layers, self.seeds, self.scaler, self.nets = layers, seeds, StandardScaler(), []
    def fit(self, X, y):
        Xs = self.scaler.fit_transform(X).astype(np.float32); y = y.astype(np.float32)
        n_val = max(1, len(Xs) // 10)                             # the last tenth in time order is the early-stopping set
        self.nets = []
        for seed in self.seeds:
            tf.keras.utils.set_random_seed(seed)
            net = keras.Sequential([keras.Input(shape=(Xs.shape[1],))] + [keras.layers.Dense(n, activation="relu") for n in self.layers] + [keras.layers.Dense(1)])
            net.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
            net.fit(Xs[:-n_val], y[:-n_val], validation_data=(Xs[-n_val:], y[-n_val:]), epochs=200, batch_size=2048, verbose=0,
                    callbacks=[keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
                               keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=4, min_lr=1e-5)])
            self.nets.append(net)
        return self
    def predict(self, X):
        Xs = self.scaler.transform(X).astype(np.float32)
        return np.mean([net.predict(Xs, batch_size=8192, verbose=0).ravel() for net in self.nets], axis=0)

# architecture screen on the sample (one seed), scored on its five folds
screen = {}
for layers in ann_candidates():
    scores = []
    for tr, va in cv_sub:
        m = ANN(layers, seeds=(0,)).fit(X_sub[tr], y_sub[tr])
        scores.append(rmse(y_sub[va] - m.predict(X_sub[va])))
    screen[layers] = float(np.mean(scores))
    print(f"ANN {layers}: sample CV RMSE {screen[layers]:.3f} dB")
best["ANN"] = min(screen, key=screen.get)
json.dump({k: (list(v) if k == "ANN" else v) for k, v in best.items()}, open("CV_Results/history_best_params.json", "w"), indent=1, default=float)
print("selected ANN architecture:", best["ANN"])

for rung in BENCH:
    t0 = time.time()
    p, models[("ANN", rung)] = oof_and_test(lambda: ANN(best["ANN"]), H[rung])
    results[("ANN", rung)] = evaluate("ANN", p)
    save_residuals(f"ANN_{rung}", p)
    print(f"ANN {rung}: hold-out RMSE {results[('ANN', rung)]['hold-out RMSE']:.3f} dB, margin 0.99 {results[('ANN', rung)]['margin 0.99']:.2f} dB  ({time.time() - t0:.0f} s)")


I0000 00:00:1791216181.139137 1325591 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791216181.162704 1325591 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1791216181.810852 1325591 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


W0000 00:00:1791216182.133128 1325591 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.


I0000 00:00:1791216182.462634 1390838 service.cc:153] XLA service 0x77fac0018ef0 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1791216182.462646 1390838 service.cc:161]   StreamExecutor [0]: Host, Default Version (Driver: 0.0.0; Runtime: 0.0.0; Toolkit: 0.0.0; DNN: 0.0.0)
I0000 00:00:1791216182.472550 1390838 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1791216182.603758 1390838 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


ANN (32, 16): sample CV RMSE 2.850 dB


...

ANN (128, 64, 32): sample CV RMSE 2.900 dB
selected ANN architecture: (32, 16)


ANN H1: hold-out RMSE 2.498 dB, margin 0.99 11.07 dB  (576 s)


ANN H4: hold-out RMSE 1.995 dB, margin 0.99 10.14 dB  (568 s)


ANN H6: hold-out RMSE 1.916 dB, margin 0.99 10.42 dB  (621 s)


ANN H7: hold-out RMSE 1.918 dB, margin 0.99 10.35 dB  (852 s)


In [8]:
# Family table on the benchmark rungs, and the full ladder of the two cheap families
table = pd.DataFrame(results).T
table.index.names = ["family", "rung"]
table.to_csv("CV_Results/history_benchmark.csv")
print("hold-out RMSE (dB) by family and rung")
print(table["hold-out RMSE"].unstack("rung").round(3).to_string())
print("\nfade margin for 99 % (dB), calibrated on folds 0-4")
print(table["margin 0.99"].unstack("rung").round(2).to_string())
print("\nreliability reached on the hold-out with that margin")
print(table["reached 0.99"].unstack("rung").round(4).to_string())


hold-out RMSE (dB) by family and rung
rung                    -     H1     H2     H3     H4     H5     H6     H7
family                                                                    
ANN                   NaN  2.498    NaN    NaN  1.995    NaN  1.916  1.918
LGBM                  NaN  2.336  2.086  2.082  1.950  1.821  1.824  1.825
Linear                NaN  2.508  2.399  2.399  2.393  2.372  2.372  2.372
RF                    NaN  2.353    NaN    NaN  1.976    NaN  1.868  1.869
XGB                   NaN  2.339    NaN    NaN  1.948    NaN  1.822  1.820
link mean           5.035    NaN    NaN    NaN    NaN    NaN    NaN    NaN
previous packet     3.044    NaN    NaN    NaN    NaN    NaN    NaN    NaN
previous-hour mean  2.712    NaN    NaN    NaN    NaN    NaN    NaN    NaN

fade margin for 99 % (dB), calibrated on folds 0-4
rung                    -     H1     H2     H3     H4     H5     H6     H7
family                                                                    
ANN       

In [9]:
# A link the model has never seen (rung H4): fit on five links with folds below 4, calibrate on their fold 4, test on the sixth link's hold-out
dev = d["device_id"].to_numpy()
X4 = X_all[H["H4"]].to_numpy()
unseen = {}
for name, make in {**factories, "ANN": lambda: ANN(best["ANN"])}.items():
    for link in sorted(np.unique(dev)):
        other, this = dev != link, dev == link
        fit_m, cal_m, test_m = ok & other & (fold >= 0) & (fold < 4), cal & other & (fold == 4), ev & this
        m = make().fit(X4[fit_m], y[fit_m])
        e_cal, e_test = y[cal_m] - m.predict(X4[cal_m]), y[test_m] - m.predict(X4[test_m])
        fm = float(np.quantile(e_cal, 0.99))
        unseen[(name, link)] = {"RMSE on the unseen link": rmse(e_test), "margin 0.99 from five links": fm, "reached on the unseen link": float((e_test <= fm).mean())}
unseen_df = pd.DataFrame(unseen).T
unseen_df.index.names = ["family", "link"]
unseen_df.to_csv("CV_Results/history_unseen_link.csv")
print("RMSE on the unseen link (dB); for comparison the H4 hold-out RMSE with the link in training is in the table above")
print(unseen_df["RMSE on the unseen link"].unstack("link").round(2).to_string())
print("\nreliability reached on the unseen link with the 99 % margin calibrated on the other five")
print(unseen_df["reached on the unseen link"].unstack("link").round(4).to_string())


RMSE on the unseen link (dB); for comparison the H4 hold-out RMSE with the link in training is in the table above
link     ED0   ED1   ED2   ED3   ED4   ED5
family                                    
ANN     2.10  2.16  2.29  1.99  2.00  2.07
LGBM    1.99  2.16  2.21  1.91  1.94  1.95
Linear  2.42  2.53  2.60  2.26  2.30  2.27
RF      2.01  2.18  2.23  1.92  1.95  1.97
XGB     1.99  2.16  2.20  1.91  1.94  1.95

reliability reached on the unseen link with the 99 % margin calibrated on the other five
link       ED0     ED1     ED2     ED3     ED4     ED5
family                                                
ANN     0.9976  0.9982  0.9937  0.9992  0.9976  0.9978
LGBM    0.9969  0.9979  0.9934  0.9988  0.9971  0.9976
Linear  0.9972  0.9980  0.9941  0.9989  0.9977  0.9980
RF      0.9972  0.9980  0.9938  0.9991  0.9972  0.9978
XGB     0.9970  0.9978  0.9935  0.9988  0.9971  0.9978


In [10]:
# Per-link hold-out RMSE at H4
per_link = {}
for name in list(factories) + ["ANN"]:
    r = pd.read_csv(f"Residuals/residuals_{name}_H4_test.csv", usecols=["device_id", "resid_db"])
    per_link[name] = r.groupby("device_id")["resid_db"].apply(lambda s: float(np.sqrt(np.mean(np.square(s)))))
per_link = pd.DataFrame(per_link)
per_link["previous-hour mean"] = pd.Series(y[ev]).groupby(dev[ev]).apply(lambda s: float(np.sqrt(np.mean(np.square(s)))))
per_link.round(3)


,Linear,RF,XGB,LGBM,ANN,previous-hour mean
device_id,,,,,,
ED0,2.421,1.953,1.927,1.929,1.944,2.752
ED1,2.522,2.118,2.088,2.090,2.123,2.744
ED2,2.592,2.155,2.119,2.120,2.187,3.008
ED3,2.249,1.853,1.836,1.839,1.876,2.431
ED4,2.295,1.878,1.848,1.849,1.898,2.651
ED5,2.253,1.875,1.847,1.848,1.917,2.651
